# 03 · Feature Comparison (configs A–F) across horizons

**Objective.** The full multi-station matrix: **5 held-out station-years × 5 horizons ×
6 feature configurations = 150 CNN runs** (+ 25 persistence baselines). This is the core
experiment behind the research question — does any exogenous input beat SWC-only?

**Provenance & reproducibility**

- **Results shown here:** ⬇ **loaded from the November-17 experiments.** The RMSE numbers come from `results/nov17_results.csv` (150 CNN runs + 25 persistence baselines, trained 2026-11-17 on Colab GPU). The aggregation tables below are computed here.
- **Experimental setup:** Held-out station-years = {FD11-2023, FD11-2020, CB06-2023, CB25-2020, FD02-2020}; horizons 24/48/72/96/168 h; model `LargerCNN`; 24 h input window; validation year 2019.
- **Implementation:** Each row was produced by `src.evaluation.run_fold(store, stations, st, yr, src.config.feature_columns(<group>), horizon=h)` — see the config table below.
- **Original code:** the 150 production CNN runs and the HP sweep were generated by the original monolithic notebook, preserved at `archive/cnn_experiments_original_monolithic.ipynb` (run live on Google Colab GPU, 2026-11-17); the notebooks here are the refactored, `src/`-based version of that same pipeline.
- **Reproduce:** GPU loop in the cell below, or a shell loop over horizons (`--horizon` takes one value per call): `for H in 24 48 72 96 168; do python main.py --horizon $H --features swc,existing_rain,lagged_rain,cumulative_rain,temperature,combined_met; done` → writes `results/experiments.csv`.

In [1]:
# --- project setup: portable across local / Google Drive / fresh Colab ---
import os, sys

def find_project_root(marker="src"):
    # 1) walk up from the current directory (local checkout or Drive upload)
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    # 2) common Colab upload locations
    for c in ("/content/soil_moisture_research",
              "/content/drive/MyDrive/soil_moisture_research"):
        if os.path.isdir(os.path.join(c, marker)):
            return c
    raise FileNotFoundError(
        "Could not find the project root (a folder containing 'src/').\n"
        "Upload the whole soil_moisture_research/ folder to Drive/Colab and "
        "open this notebook from inside notebooks/.")

ROOT = find_project_root()
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("project root set ->", os.path.basename(ROOT))  # portable: no absolute path

project root set -> soil_moisture_research


### The six feature configurations
Each configuration always includes `SWC_20` history (column 0) and adds an exogenous group
on top. The exact columns come from `src.config.FEATURE_GROUPS`:

| cfg | config key (`FEATURE_GROUPS`) | exogenous columns added to `SWC_20` |
|----|----|----|
| **A** | `swc` | *(none — baseline)* |
| **B** | `existing_rain` | `Ppt_log` |
| **C** | `lagged_rain` | `Ppt_log`, `Ppt_log_lag1`, `Ppt_log_lag3`, `Ppt_log_lag6` |
| **D** | `cumulative_rain` | `Ppt_6h`, `Ppt_24h`, `Ppt_72h` |
| **E** | `temperature` | `Tair` |
| **F** | `combined_met` | `Ppt_log`, `Tair`, `RH`, `sin_doy`, `cos_doy` |

`src.config.feature_columns('existing_rain')`, for example, returns
`['SWC_20', 'Ppt_log']`, which is exactly what `run_fold` scales, windows, and trains on.

In [2]:
import pandas as pd
res = pd.read_csv('results/nov17_results.csv')
print('rows:', len(res), '| configs:', res.config.nunique(), '| horizons:', sorted(res.horizon.unique()))
res.groupby('config').RMSE.mean().round(4)

rows: 175 | configs: 7 | horizons: [np.int64(24), np.int64(48), np.int64(72), np.int64(96), np.int64(168)]


config
A: SWC_20 only          0.0234
B: + existing rain      0.0236
C: + lagged rain        0.0238
D: + cumulative rain    0.0236
E: + air temperature    0.0236
F: combined met         0.0238
persistence             0.0235
Name: RMSE, dtype: float64

### Regenerating these rows (optional, GPU recommended)
```python
from src.data_processing import load_dataframe, build_station_year_store
from src.evaluation import run_fold
from src import config as C
store = build_station_year_store(load_dataframe(), C.SELECTED_STATIONS)
for (st, yr) in C.TEST_STATION_YEARS:
    for h in [24, 48, 72, 96, 168]:
        for grp in ['swc','existing_rain','lagged_rain','cumulative_rain','temperature','combined_met']:
            row, _ = run_fold(store, C.SELECTED_STATIONS, st, yr, C.feature_columns(grp), horizon=h)
```
Or from a shell: `python main.py --horizon 72 --features swc,existing_rain,lagged_rain`.

In [3]:
# Mean %RMSE improvement vs SWC-only (config A), by config and horizon.
# Negative = the exogenous feature made the forecast WORSE than SWC-only.
cnn = res[res.architecture=='LargerCNN']
cnn.pivot_table(index='config', columns='horizon', values='impr_vs_swc_%', aggfunc='mean').round(2)

horizon,24,48,72,96,168
config,,,,,
A: SWC_20 only,0.00,0.00,0.00,0.00,0.00
B: + existing rain,-2.98,-1.65,-1.90,-1.70,0.58
C: + lagged rain,-6.43,-5.00,-1.65,-1.94,0.45
D: + cumulative rain,-3.95,-2.31,-2.57,-1.33,1.68
E: + air temperature,-2.48,-2.01,-1.26,-1.27,-0.07
F: combined met,-11.71,-5.48,-2.01,-1.70,1.61


**Reading this table.** Almost every cell is negative: at short horizons adding exogenous
features *hurts*, and only at the 168 h horizon do a few configs edge slightly positive.
Notebook **07** aggregates this into the formal feature-importance discussion.